In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from pathlib import Path
import json
from typing import Dict, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

## 1. Focal Loss Implementation

Focal loss to handle class imbalance in action anticipation.

In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss for addressing class imbalance (Lin et al., 2017)"""
    
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0, reduction: str = 'mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        Args:
            inputs: (batch_size, num_classes) - logits
            targets: (batch_size,) - class indices
        """
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        p_t = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - p_t) ** self.gamma * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

## 2. Attentive Probe Architecture

Multi-query attentive probe with 3 learnable query tokens for action, verb, and noun prediction.

In [ ]:
class AttentiveAnticipationProbe(nn.Module):
    """Attentive probe for action anticipation with 3 query tokens"""
    
    def __init__(
        self,
        embed_dim: int = 768,
        num_heads: int = 12,
        num_layers: int = 4,
        num_action_classes: int = 3806,  # EPIC-Kitchen 100
        num_verb_classes: int = 97,
        num_noun_classes: int = 300,
        dropout: float = 0.1
    ):
        super().__init__()
        
        self.embed_dim = embed_dim
        self.num_queries = 3  # action, verb, noun
        
        # Learnable query tokens
        self.query_tokens = nn.Parameter(torch.randn(1, self.num_queries, embed_dim))
        
        # Shared attention blocks
        self.attention_layers = nn.ModuleList([
            nn.TransformerDecoderLayer(
                d_model=embed_dim,
                nhead=num_heads,
                dim_feedforward=embed_dim * 4,
                dropout=dropout,
                activation='gelu',
                batch_first=True
            )
            for _ in range(num_layers)
        ])
        
        # Layer norm
        self.norm = nn.LayerNorm(embed_dim)
        
        # Separate linear classifiers for each prediction task
        self.action_classifier = nn.Linear(embed_dim, num_action_classes)
        self.verb_classifier = nn.Linear(embed_dim, num_verb_classes)
        self.noun_classifier = nn.Linear(embed_dim, num_noun_classes)
        
        self._init_weights()
    
    def _init_weights(self):
        nn.init.trunc_normal_(self.query_tokens, std=0.02)
        nn.init.normal_(self.action_classifier.weight, std=0.01)
        nn.init.normal_(self.verb_classifier.weight, std=0.01)
        nn.init.normal_(self.noun_classifier.weight, std=0.01)
        nn.init.zeros_(self.action_classifier.bias)
        nn.init.zeros_(self.verb_classifier.bias)
        nn.init.zeros_(self.noun_classifier.bias)
    
    def forward(self, encoder_predictor_tokens: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Args:
            encoder_predictor_tokens: (batch_size, num_tokens, embed_dim)
                Concatenated encoder and predictor outputs
        
        Returns:
            action_logits: (batch_size, num_action_classes)
            verb_logits: (batch_size, num_verb_classes)
            noun_logits: (batch_size, num_noun_classes)
        """
        batch_size = encoder_predictor_tokens.shape[0]
        
        # Expand query tokens for batch
        queries = self.query_tokens.expand(batch_size, -1, -1)
        
        # Pass through attention layers
        for layer in self.attention_layers:
            queries = layer(queries, encoder_predictor_tokens)
        
        queries = self.norm(queries)
        
        # Extract each query output
        action_query = queries[:, 0]  # First query for action
        verb_query = queries[:, 1]    # Second query for verb
        noun_query = queries[:, 2]    # Third query for noun
        
        # Apply classifiers
        action_logits = self.action_classifier(action_query)
        verb_logits = self.verb_classifier(verb_query)
        noun_logits = self.noun_classifier(noun_query)
        
        return action_logits, verb_logits, noun_logits

## 3. V-JEPA 2 Wrapper (Encoder + Predictor)

Frozen encoder and predictor components.

In [ ]:
class VJEPA2AnticipationModel(nn.Module):
    """Complete model: Frozen V-JEPA 2 + Trainable Anticipation Probe"""
    
    def __init__(
        self,
        vjepa_encoder,
        vjepa_predictor,
        probe_config: Optional[Dict] = None
    ):
        super().__init__()
        
        # Frozen V-JEPA 2 components
        self.encoder = vjepa_encoder
        self.predictor = vjepa_predictor
        
        # Freeze encoder and predictor
        for param in self.encoder.parameters():
            param.requires_grad = False
        for param in self.predictor.parameters():
            param.requires_grad = False
        
        # Trainable anticipation probe
        probe_config = probe_config or {}
        self.probe = AttentiveAnticipationProbe(**probe_config)
        
        self.encoder.eval()
        self.predictor.eval()
    
    def forward(
        self,
        video_context: torch.Tensor,
        future_mask_tokens: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Args:
            video_context: (batch_size, num_frames, C, H, W)
                Video clip ending 1 second before action
            future_mask_tokens: Mask tokens for future frame prediction
        
        Returns:
            action_logits, verb_logits, noun_logits
        """
        with torch.no_grad():
            # Encode video context
            encoder_output = self.encoder(video_context)
            
            # Predict future representations
            predictor_output = self.predictor(
                encoder_output,
                future_mask_tokens
            )
            
            # Concatenate along token dimension
            combined_tokens = torch.cat([encoder_output, predictor_output], dim=1)
        
        # Pass through trainable probe
        action_logits, verb_logits, noun_logits = self.probe(combined_tokens)
        
        return action_logits, verb_logits, noun_logits

## 4. EPIC Kitchen Dataset Loader

Dataset class for loading video clips with anticipation annotations.

In [ ]:
class EPICKitchenAnticipationDataset(Dataset):
    """EPIC Kitchen dataset for action anticipation"""
    
    def __init__(
        self,
        data_root: str,
        annotation_file: str,
        anticipation_time: float = 1.0,  # seconds
        clip_duration: float = 2.0,  # seconds of context
        fps: int = 30,
        transform=None,
        split: str = 'train'
    ):
        self.data_root = Path(data_root)
        self.anticipation_time = anticipation_time
        self.clip_duration = clip_duration
        self.fps = fps
        self.transform = transform
        self.split = split
        
        # Load annotations
        with open(annotation_file, 'r') as f:
            self.annotations = json.load(f)
        
        # Calculate frame indices
        self.anticipation_frames = int(anticipation_time * fps)
        self.context_frames = int(clip_duration * fps)
    
    def __len__(self):
        return len(self.annotations)
    
    def __getitem__(self, idx: int) -> Dict:
        ann = self.annotations[idx]
        
        # Extract annotation info
        video_id = ann['video_id']
        action_start_frame = ann['start_frame']
        action_class = ann['action_class']
        verb_class = ann['verb_class']
        noun_class = ann['noun_class']
        
        # Calculate context window: ends 1 second before action
        context_end_frame = action_start_frame - self.anticipation_frames
        context_start_frame = max(0, context_end_frame - self.context_frames)
        
        # Load video frames (placeholder - implement actual video loading)
        video_frames = self._load_video_frames(
            video_id,
            context_start_frame,
            context_end_frame
        )
        
        if self.transform:
            video_frames = self.transform(video_frames)
        
        return {
            'video': video_frames,
            'action_class': action_class,
            'verb_class': verb_class,
            'noun_class': noun_class,
            'video_id': video_id
        }
    
    def _load_video_frames(self, video_id: str, start_frame: int, end_frame: int):
        """Load video frames - implement based on your data format"""
        # Placeholder implementation
        # Replace with actual video loading (e.g., using decord, torchvision, etc.)
        num_frames = end_frame - start_frame
        # Return dummy tensor for now
        return torch.randn(num_frames, 3, 224, 224)

## 5. Training Loop

Training procedure with focal loss for each classifier.

In [ ]:
def train_anticipation_probe(
    model: VJEPA2AnticipationModel,
    train_loader: DataLoader,
    val_loader: DataLoader,
    num_epochs: int = 50,
    learning_rate: float = 1e-4,
    device: str = 'cuda',
    focal_alpha: float = 0.25,
    focal_gamma: float = 2.0,
    save_dir: str = './checkpoints'
):
    """Train the anticipation probe on EPIC Kitchen dataset"""
    
    model = model.to(device)
    model.train()
    
    # Only optimize probe parameters (encoder and predictor are frozen)
    optimizer = torch.optim.AdamW(
        model.probe.parameters(),
        lr=learning_rate,
        weight_decay=0.05
    )
    
    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_epochs
    )
    
    # Focal loss for each classifier
    action_criterion = FocalLoss(alpha=focal_alpha, gamma=focal_gamma)
    verb_criterion = FocalLoss(alpha=focal_alpha, gamma=focal_gamma)
    noun_criterion = FocalLoss(alpha=focal_alpha, gamma=focal_gamma)
    
    best_val_acc = 0.0
    save_path = Path(save_dir)
    save_path.mkdir(parents=True, exist_ok=True)
    
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0
        action_correct = 0
        verb_correct = 0
        noun_correct = 0
        total_samples = 0
        
        for batch_idx, batch in enumerate(train_loader):
            video = batch['video'].to(device)
            action_labels = batch['action_class'].to(device)
            verb_labels = batch['verb_class'].to(device)
            noun_labels = batch['noun_class'].to(device)
            
            # Create future mask tokens (placeholder - adjust based on V-JEPA 2 implementation)
            batch_size = video.shape[0]
            future_mask_tokens = torch.zeros(batch_size, 1, model.probe.embed_dim).to(device)
            
            # Forward pass
            action_logits, verb_logits, noun_logits = model(video, future_mask_tokens)
            
            # Calculate losses independently
            loss_action = action_criterion(action_logits, action_labels)
            loss_verb = verb_criterion(verb_logits, verb_labels)
            loss_noun = noun_criterion(noun_logits, noun_labels)
            
            # Sum losses before backpropagation
            loss = loss_action + loss_verb + loss_noun
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.probe.parameters(), max_norm=1.0)
            optimizer.step()
            
            # Calculate accuracies
            action_pred = action_logits.argmax(dim=1)
            verb_pred = verb_logits.argmax(dim=1)
            noun_pred = noun_logits.argmax(dim=1)
            
            action_correct += (action_pred == action_labels).sum().item()
            verb_correct += (verb_pred == verb_labels).sum().item()
            noun_correct += (noun_pred == noun_labels).sum().item()
            total_samples += batch_size
            total_loss += loss.item()
            
            if (batch_idx + 1) % 50 == 0:
                print(f"Epoch [{epoch+1}/{num_epochs}] Batch [{batch_idx+1}/{len(train_loader)}] "
                      f"Loss: {loss.item():.4f} "
                      f"Action Acc: {100*action_correct/total_samples:.2f}% "
                      f"Verb Acc: {100*verb_correct/total_samples:.2f}% "
                      f"Noun Acc: {100*noun_correct/total_samples:.2f}%")
        
        # Validation
        val_metrics = validate(model, val_loader, device)
        
        print(f"\nEpoch [{epoch+1}/{num_epochs}] Summary:")
        print(f"Train Loss: {total_loss/len(train_loader):.4f}")
        print(f"Val Action Acc: {val_metrics['action_acc']:.2f}%")
        print(f"Val Verb Acc: {val_metrics['verb_acc']:.2f}%")
        print(f"Val Noun Acc: {val_metrics['noun_acc']:.2f}%\n")
        
        scheduler.step()
        
        # Save best model
        if val_metrics['action_acc'] > best_val_acc:
            best_val_acc = val_metrics['action_acc']
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.probe.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_action_acc': val_metrics['action_acc'],
                'val_verb_acc': val_metrics['verb_acc'],
                'val_noun_acc': val_metrics['noun_acc'],
            }, save_path / 'best_probe.pt')
            print(f"Saved best model with action accuracy: {best_val_acc:.2f}%")


def validate(model: VJEPA2AnticipationModel, val_loader: DataLoader, device: str) -> Dict:
    """Validate the model"""
    model.eval()
    action_correct = 0
    verb_correct = 0
    noun_correct = 0
    total_samples = 0
    
    with torch.no_grad():
        for batch in val_loader:
            video = batch['video'].to(device)
            action_labels = batch['action_class'].to(device)
            verb_labels = batch['verb_class'].to(device)
            noun_labels = batch['noun_class'].to(device)
            
            batch_size = video.shape[0]
            future_mask_tokens = torch.zeros(batch_size, 1, model.probe.embed_dim).to(device)
            
            action_logits, verb_logits, noun_logits = model(video, future_mask_tokens)
            
            action_pred = action_logits.argmax(dim=1)
            verb_pred = verb_logits.argmax(dim=1)
            noun_pred = noun_logits.argmax(dim=1)
            
            action_correct += (action_pred == action_labels).sum().item()
            verb_correct += (verb_pred == verb_labels).sum().item()
            noun_correct += (noun_pred == noun_labels).sum().item()
            total_samples += batch_size
    
    model.train()
    
    return {
        'action_acc': 100 * action_correct / total_samples,
        'verb_acc': 100 * verb_correct / total_samples,
        'noun_acc': 100 * noun_correct / total_samples,
    }

## 6. Example Usage

Setup and run the anticipation probe training.

In [ ]:
# Configuration
config = {
    'data_root': '/path/to/epic_kitchen/videos',
    'train_annotations': '/path/to/epic_kitchen/train_annotations.json',
    'val_annotations': '/path/to/epic_kitchen/val_annotations.json',
    'batch_size': 16,
    'num_workers': 4,
    'num_epochs': 50,
    'learning_rate': 1e-4,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}

# Probe configuration for EPIC Kitchen 100
probe_config = {
    'embed_dim': 768,  # Adjust based on V-JEPA 2 model
    'num_heads': 12,
    'num_layers': 4,
    'num_action_classes': 3806,  # EPIC-Kitchen 100
    'num_verb_classes': 97,
    'num_noun_classes': 300,
    'dropout': 0.1
}

print(f"Using device: {config['device']}")
print(f"Probe config: {probe_config}")

In [ ]:
# Load V-JEPA 2 encoder and predictor (placeholder)
# Replace with actual V-JEPA 2 model loading
# vjepa_encoder = load_vjepa2_encoder()
# vjepa_predictor = load_vjepa2_predictor()

# For demonstration purposes, create dummy models
class DummyEncoder(nn.Module):
    def forward(self, x):
        batch_size = x.shape[0]
        return torch.randn(batch_size, 196, 768).to(x.device)  # (B, num_patches, embed_dim)

class DummyPredictor(nn.Module):
    def forward(self, encoder_output, mask_tokens):
        batch_size = encoder_output.shape[0]
        return torch.randn(batch_size, 16, 768).to(encoder_output.device)  # (B, num_mask_tokens, embed_dim)

vjepa_encoder = DummyEncoder()
vjepa_predictor = DummyPredictor()

print("Loaded V-JEPA 2 encoder and predictor (frozen)")

In [ ]:
# Create full model
model = VJEPA2AnticipationModel(
    vjepa_encoder=vjepa_encoder,
    vjepa_predictor=vjepa_predictor,
    probe_config=probe_config
)

# Count trainable parameters
trainable_params = sum(p.numel() for p in model.probe.parameters() if p.requires_grad)
frozen_params = sum(p.numel() for p in model.encoder.parameters()) + \
                sum(p.numel() for p in model.predictor.parameters())

print(f"Trainable parameters (probe): {trainable_params:,}")
print(f"Frozen parameters (encoder + predictor): {frozen_params:,}")

In [ ]:
# Create datasets and dataloaders
# NOTE: Update paths to your actual EPIC Kitchen dataset

# train_dataset = EPICKitchenAnticipationDataset(
#     data_root=config['data_root'],
#     annotation_file=config['train_annotations'],
#     split='train'
# )

# val_dataset = EPICKitchenAnticipationDataset(
#     data_root=config['data_root'],
#     annotation_file=config['val_annotations'],
#     split='validation'
# )

# train_loader = DataLoader(
#     train_dataset,
#     batch_size=config['batch_size'],
#     shuffle=True,
#     num_workers=config['num_workers'],
#     pin_memory=True
# )

# val_loader = DataLoader(
#     val_dataset,
#     batch_size=config['batch_size'],
#     shuffle=False,
#     num_workers=config['num_workers'],
#     pin_memory=True
# )

print("Dataset loaders ready (commented out - update paths)")

In [ ]:
# Start training
# Uncomment when datasets are ready

# train_anticipation_probe(
#     model=model,
#     train_loader=train_loader,
#     val_loader=val_loader,
#     num_epochs=config['num_epochs'],
#     learning_rate=config['learning_rate'],
#     device=config['device'],
#     focal_alpha=0.25,
#     focal_gamma=2.0,
#     save_dir='./checkpoints/epic_anticipation'
# )

print("Training setup complete. Uncomment to start training.")

## 7. Evaluation Metrics

Additional evaluation metrics for action anticipation.

In [ ]:
def compute_metrics(predictions: Dict, targets: Dict) -> Dict:
    """Compute evaluation metrics for action anticipation"""
    
    action_preds = predictions['action']
    verb_preds = predictions['verb']
    noun_preds = predictions['noun']
    
    action_targets = targets['action']
    verb_targets = targets['verb']
    noun_targets = targets['noun']
    
    # Top-1 accuracy
    action_top1 = (action_preds == action_targets).float().mean() * 100
    verb_top1 = (verb_preds == verb_targets).float().mean() * 100
    noun_top1 = (noun_preds == noun_targets).float().mean() * 100
    
    # Combined verb-noun accuracy
    verb_noun_correct = ((verb_preds == verb_targets) & (noun_preds == noun_targets)).float()
    verb_noun_top1 = verb_noun_correct.mean() * 100
    
    return {
        'action_top1': action_top1.item(),
        'verb_top1': verb_top1.item(),
        'noun_top1': noun_top1.item(),
        'verb_noun_top1': verb_noun_top1.item()
    }

print("Evaluation metrics defined")

## Notes

### Implementation Details:
1. **Video Sampling**: Clips end exactly 1 second before action start
2. **Future Prediction**: Predictor targets frames 1 second into the future
3. **Frozen Components**: Encoder and predictor weights are not updated
4. **Probe Training**: Only the attentive probe is trainable
5. **Multi-task Learning**: Three independent classifiers with separate focal losses

### TODO:
- [ ] Implement actual V-JEPA 2 encoder/predictor loading
- [ ] Implement video frame loading based on your data format
- [ ] Add data augmentation for video clips
- [ ] Implement top-5 accuracy metrics
- [ ] Add learning rate warmup
- [ ] Add mixed precision training support
- [ ] Update paths to EPIC Kitchen dataset
- [ ] Fine-tune hyperparameters (see Appendix D.1 in paper)